# LLM fuzzy-unification: composition-cell gap-filling on CLUTRR kinship

**Artifact `art_OvidVcsfr5HM`** — a *neuro-symbolic* division of labour for reasoning over
text. A **sound symbolic closure engine** does the multi-hop chaining; a probabilistic **LLM
reader** is invoked *only* to fill the one missing/ambiguous cell of the composition table
where strict symbolic matching fails (the "fuzzy unification" step).

This demo reproduces the **HEADLINE result (Setting 2, CLUTRR kinship)**: under a
converse-closed *ablated* kinship table, solvable queries that needed an ablated rule become
**gaps** where the symbolic engine abstains. We compare four systems on the same query pool:

| system | what it is |
|---|---|
| **symbolic** | pure closure — never hallucinates, abstains on gaps |
| **Mode-P** (cell-fill) | neuro-symbolic: LLM supplies only the missing *atomic* rule, symbolic does the chaining |
| **Mode-S** (story-fill) | neuro-symbolic: LLM does the whole multi-hop read from the story |
| **raw LLM** | pure-neural chain-of-thought, answers every query directly |

**The point:** Mode-P recovers 100% of gaps at precision 1.00 and **cuts the confident-wrong
(hallucination) rate to 0**, while the raw LLM hallucinates on ~half its answers — the right
neuro-symbolic division of labour. Naive story-level filling (Mode-S) is the *honest negative*.

> This notebook re-runs the experiment's **analysis** on the **precomputed, sha256-cached LLM
> predictions** (the live reader is `google/gemini-3.1-flash-lite`, temp 0; the cache made the
> full run cost \$0.28). No API key, GPU, or dataset download is needed — it loads a curated
> 100-query subset and recomputes coverage / accuracy / hallucination-rate / risk-coverage
> with the **original scoring functions**, then renders the auditable trace-graphs.

## Setup

### Install dependencies
Only `numpy` (analysis) and `matplotlib` (plots) are needed — both pre-installed on Colab.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# numpy + matplotlib are pre-installed on Colab (install locally to match Colab versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

### Imports
The original `method.py` additionally imports the live pipeline modules
(`kinship`, `llm`, `qcn.algebras`, `stats`, `dataio`) that run the symbolic engine and call
the LLM. This demo re-analyses the **cached predictions**, so it only needs the analysis stack.

In [ ]:
import json, os
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

### Data loading
Load the curated subset from GitHub (with a local fallback for offline use).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-40a89b-no-derivation-no-relation-a-closure-cert/main/round-4/experiment-2/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("loaded:", data["metadata"]["method_name"])
print("dataset:", data["datasets"][0]["dataset"],
      "| examples in subset:", len(data["datasets"][0]["examples"]))
print("reference full pool size:", data["metadata"]["clutrr_gap_pool_counts"]["n_mixed_pool"],
      "| headline ablation K =", data["metadata"]["headline_K"])

## Config

All tunable knobs live here. They start small; raise them for a fuller re-analysis. The whole
notebook runs in well under a second at any setting (it only re-aggregates cached predictions).

- `N_EXAMPLES` — how many of the loaded queries to re-analyze (subset holds 100; full pool = 663).
- `TAUS` — confidence thresholds swept for the risk-coverage curve.
- `NET_FAITHFUL_BAR` — recovered precision must exceed this to count as "net faithful" (paper = 0.50).
- `N_TRACES_SHOW` — number of auditable trace-graphs to render.

In [ ]:
# === Config (minimum demo values; raise for a fuller re-analysis) ===
N_EXAMPLES       = 100                       # full subset = 100; full experiment pool = 663
TAUS             = [i / 20 for i in range(21)]  # 0.00 .. 1.00 step 0.05  (paper sweep)
NET_FAITHFUL_BAR = 0.50                      # recovered-precision bar (matches method.py)
N_TRACES_SHOW    = 5                         # auditable trace-graphs to print

# --- original-scale reference (commented; reproduce by deploying the full cache) ---
# N_EXAMPLES = 663   # the full mixed pool used in the paper headline

## Reconstruct per-query records

Each `clutrr_gapfill` example carries the prediction **and** correctness flag for all four
systems. We rebuild the record structure the original analysis functions expect:
`r[method] = {"pred", "answered", "correct", "conf"}`. (`symbolic` and `Mode-P` cell-fill are
exact-table steps → confidence 1.0; `Mode-S` / `raw` carry the LLM's reported confidence.)

In [ ]:
examples = data["datasets"][0]["examples"][:N_EXAMPLES]

def _conf(ex, key, answered):
    if not answered:
        return 0.0
    if key == "S":
        return float(ex["metadata_conf_S"])
    if key == "raw":
        return float(ex["metadata_conf_raw"])
    return 1.0  # symbolic + Mode-P cell-fill: exact-table confident

records = []
for ex in examples:
    rec = {"doc_id": ex["metadata_doc_id"], "qsrc": ex["metadata_qsrc"],
           "qtgt": ex["metadata_qtgt"], "gold": ex["output"],
           "gap_origin": ex["metadata_gap_origin"], "gap_kind": ex["metadata_gap_kind"]}
    for key, pred_field, corr_field in (
            ("sym", "predict_symbolic",  "metadata_correct_symbolic"),
            ("P",   "predict_gapfill_P", "metadata_correct_P"),
            ("S",   "predict_gapfill_S", "metadata_correct_S"),
            ("raw", "predict_raw_llm",   "metadata_correct_raw")):
        pred = ex[pred_field]
        answered = (pred != "ABSTAIN")
        rec[key] = {"pred": pred, "answered": answered,
                    "correct": bool(ex[corr_field]), "conf": _conf(ex, key, answered)}
    records.append(rec)

n_manuf = sum(1 for r in records if r["gap_origin"] == "manufactured")
n_norm  = sum(1 for r in records if r["gap_origin"] == "none")
print(f"reconstructed {len(records)} per-query records "
      f"({n_manuf} manufactured gaps + {n_norm} normal queries)")

## Full-coverage comparison  (`_modesummary`, verbatim from `method.py`)

For each system: **coverage** (fraction answered), **accuracy among answered**, and the
load-bearing **confident-wrong rate** (a wrong, confident singleton = a silent-wrong-narrowing
hallucination). The symbolic engine is hallucination-safe *by construction* on this pool.

In [ ]:
def _modesummary(recs, key):
    if not recs:
        return {"n_pool": 0, "coverage": 0.0, "acc_among_answered": None,
                "confident_wrong_rate": 0.0, "n_answered": 0, "n_correct": 0, "n_wrong": 0}
    answered = [r for r in recs if r[key]["answered"]]
    correct = [r for r in answered if r[key]["correct"]]
    wrong = [r for r in answered if not r[key]["correct"]]
    npool = len(recs)
    return {
        "n_pool": npool, "coverage": len(answered) / npool,
        "acc_among_answered": (len(correct) / len(answered)) if answered else None,
        "confident_wrong_rate": len(wrong) / npool,
        "n_answered": len(answered), "n_correct": len(correct), "n_wrong": len(wrong),
    }

# full-coverage point per method (mirrors metadata.clutrr_full_coverage_point)
full_pt = {m: _modesummary(records, k) for m, k in
           (("symbolic", "sym"), ("gapfill_P", "P"), ("gapfill_S", "S"), ("raw_llm", "raw"))}
for m, s in full_pt.items():
    acc = "n/a" if s["acc_among_answered"] is None else f"{s['acc_among_answered']:.3f}"
    print(f"{m:14s} coverage={s['coverage']:.3f}  acc_among_answered={acc:>5s}  "
          f"confident_wrong={s['confident_wrong_rate']:.3f}")

## Hallucination reduction  (verbatim from `method.py`)

The quantified hallucination reduction of each neuro-symbolic mode vs the raw LLM.

In [ ]:
cw_raw = full_pt["raw_llm"]["confident_wrong_rate"]
cw_S = full_pt["gapfill_S"]["confident_wrong_rate"]
cw_P = full_pt["gapfill_P"]["confident_wrong_rate"]
hallucination_reduction = {
    "confident_wrong_rate_raw_llm": cw_raw,
    "confident_wrong_rate_gapfill_S": cw_S,
    "confident_wrong_rate_gapfill_P": cw_P,
    "confident_wrong_rate_symbolic": full_pt["symbolic"]["confident_wrong_rate"],
    "abs_reduction_S_vs_raw": cw_raw - cw_S,
    "rel_reduction_S_vs_raw": (cw_raw - cw_S) / cw_raw if cw_raw else None,
    "abs_reduction_P_vs_raw": cw_raw - cw_P,
    "rel_reduction_P_vs_raw": (cw_raw - cw_P) / cw_raw if cw_raw else None,
}
relP = hallucination_reduction["rel_reduction_P_vs_raw"]
print(f"Mode-P (cell-fill) cuts the confident-wrong (hallucination) rate by "
      f"{100*relP:.0f}% vs raw LLM ({cw_raw:.3f} -> {cw_P:.3f}) at FULL coverage")
print(f"Mode-S (story-fill) reduction vs raw LLM: "
      f"{100*hallucination_reduction['rel_reduction_S_vs_raw']:.0f}%  (the honest negative)")

## Risk-coverage curve  (`_risk_coverage_curve`, verbatim from `method.py`)

Selective accuracy as the confidence threshold sweeps `TAUS`. The neuro-symbolic Mode-P sits at
the top-right (full coverage, perfect accuracy); the raw LLM trades coverage for accuracy poorly.

In [ ]:
def _risk_coverage_curve(records, method_key, taus):
    n = len(records)
    pts = []
    for tau in taus:
        covered = [r for r in records if r[method_key]["answered"] and r[method_key]["conf"] >= tau]
        cov = len(covered) / n if n else 0.0
        acc = float(np.mean([r[method_key]["correct"] for r in covered])) if covered else None
        pts.append({"tau": round(tau, 3), "coverage": cov, "selective_accuracy": acc,
                    "n_covered": len(covered)})
    return pts

mixed_curve = {
    "symbolic_abstain": _risk_coverage_curve(records, "sym", TAUS),
    "gapfill_P": _risk_coverage_curve(records, "P", TAUS),
    "gapfill_S": _risk_coverage_curve(records, "S", TAUS),
    "raw_llm": _risk_coverage_curve(records, "raw", TAUS),
}
for name, pts in mixed_curve.items():
    p0 = pts[0]  # tau=0 (full coverage)
    acc = "n/a" if p0["selective_accuracy"] is None else f"{p0['selective_accuracy']:.3f}"
    print(f"{name:18s} @tau=0: coverage={p0['coverage']:.3f} selective_accuracy={acc}")

## Auditable trace-graphs

The deliverable's human-auditable reasoning traces. Every composition step is tagged
**`EXACT`** (exact-table closure) vs **`FUZZY`** (an LLM-resolved cell). Manufactured gaps show
a single filled cell inside an otherwise-exact chain; natural-conflict gaps show the exact-table
conflict plus one LLM story-disambiguation step (including the honest mother-vs-mother-in-law
case).

In [ ]:
traces = data["metadata"]["flagged_fuzzy_step_traces"][:N_TRACES_SHOW]
for tr in traces:
    print("=" * 80)
    print(f"doc {tr['doc_id'][:8]}  {tr['qsrc']} -> {tr['qtgt']}  gold={tr['gold']}  "
          f"({tr['gap_origin']} {tr['gap_kind']} gap)")
    print(f"  story: {tr['story'][:150]}...")
    for i, st in enumerate(tr["trace"], 1):
        if st["source"] == "exact_table":
            print(f"  step {i} [EXACT] {st.get('a')} -{st.get('t1')}-> {st.get('b')} "
                  f"-{st.get('t2')}-> {st.get('c')}   =>  {st.get('t3')}")
        elif st.get("kind") == "story_disambiguation":
            print(f"  step {i} [FUZZY] story-disambiguation  candidates={st.get('candidates')} "
                  f"-> pick={st.get('llm_pick')} (conf={st.get('llm_confidence')})")
        else:
            print(f"  step {i} [FUZZY] {st.get('a')} {st.get('t1')};{st.get('t2')} => "
                  f"{st.get('t3')}   llm={st.get('llm_predicted_t3')} "
                  f"conf={st.get('llm_confidence')} matches_true_cell={st.get('matches_true_cell')}")
    print(f"  FINAL: {tr['final_answer']}   correct={tr['correct']}   "
          f"(llm_fuzzy_steps={tr['n_llm_fuzzy_steps']})")

## Results & visualization

Left: confident-wrong (hallucination) rate per system — demo subset vs the full-pool reference.
Mode-P cell-fill drives it to **0** while keeping full coverage. Right: the risk-coverage
trade-off. The table compares the demo-subset re-analysis with the paper's full-pool numbers.

In [ ]:
ref_full = data["metadata"]["clutrr_full_coverage_point"]
methods = ["symbolic", "gapfill_P", "gapfill_S", "raw_llm"]
labels  = ["symbolic\n(abstain)", "Mode-P\n(cell-fill)", "Mode-S\n(story-fill)", "raw LLM\n(CoT)"]

# --- comparison table: demo subset vs full-pool reference ---
print(f"{'system':14s}{'cov':>7s}{'acc@ans':>9s}{'confW':>8s}   |  full-pool: "
      f"{'cov':>6s}{'acc@ans':>9s}{'confW':>8s}")
print("-" * 84)
for m in methods:
    s, r = full_pt[m], ref_full[m]
    sa = "n/a" if s["acc_among_answered"] is None else f"{s['acc_among_answered']:.2f}"
    ra = "n/a" if r["acc_among_answered"] is None else f"{r['acc_among_answered']:.2f}"
    print(f"{m:14s}{s['coverage']:>7.2f}{sa:>9s}{s['confident_wrong_rate']:>8.2f}   |"
          f"{r['coverage']:>13.2f}{ra:>9s}{r['confident_wrong_rate']:>8.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# left: hallucination bar (demo subset vs full-pool reference)
demo_cw = [full_pt[m]["confident_wrong_rate"] for m in methods]
ref_cw  = [ref_full[m]["confident_wrong_rate"] for m in methods]
x = np.arange(len(methods)); w = 0.38
axes[0].bar(x - w/2, demo_cw, w, label=f"demo subset (n={len(records)})", color="#d62728")
axes[0].bar(x + w/2, ref_cw,  w, label="full pool (n=663)", color="#9467bd", alpha=.75)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_ylabel("confident-wrong (hallucination) rate")
axes[0].set_title("Hallucination rate by system\nMode-P cell-fill -> 0")
axes[0].legend(); axes[0].set_ylim(0, max(ref_cw + demo_cw) * 1.25 + 0.01)

# right: risk-coverage curve
for disp, key in (("symbolic", "symbolic_abstain"), ("Mode-P", "gapfill_P"),
                  ("Mode-S", "gapfill_S"), ("raw LLM", "raw_llm")):
    pts = [p for p in mixed_curve[key] if p["selective_accuracy"] is not None]
    if pts:
        axes[1].plot([p["coverage"] for p in pts], [p["selective_accuracy"] for p in pts],
                     "o-", label=disp, markersize=5)
axes[1].set_xlabel("coverage"); axes[1].set_ylabel("selective accuracy")
axes[1].set_title("Risk-coverage (selective accuracy vs coverage)")
axes[1].legend(); axes[1].grid(alpha=.3); axes[1].set_ylim(0, 1.05)

plt.tight_layout(); plt.savefig("demo_results.png", dpi=110); plt.show()

v = data["metadata"]["verdict"]
print(f"\nVERDICT (full run): adds net-faithful coverage on CLUTRR = "
      f"{v['adds_net_faithful_coverage_clutrr']} "
      f"(best mode={v['best_mode']}, recovered precision={v['recovered_precision']}, "
      f"net +{v['net_recovered_correct_minus_wrong']})")
print("kinship composition-cell accuracy (Mode-P):",
      data["metadata"]["kinship_composition_cell_accuracy"]["exact_match_acc"])